# HW3: Atomic cashflows and the Visitor pattern

This assignment develops two skills:

1. **Model cashflow contracts.** Implement `ProductFixedAccruedCashflow` and `ProductOvernightIndexCashflow` using the provided `Product` and `ProductCashflow` bases. Understand their contractual fields, including dates, notional, currency, accrual conventions, index and spread. Each object is a building block for a larger product, such as a swap.
2. **Use the Visitor pattern.** Collect and display each product's attributes through a visitor. This demonstrates how to add an operation without changing the product classes.

Read [IV. Visitor Pattern](https://immense-chocolate-92e.notion.site/IV-Visitor-Pattern-2d4831d0b1d58191bc36ebae3d771d58); implement `accept()` and the display visitor to produce the tables below.

Complete the six `#TODO` blocks in `fixedincomelib/product/linear_products.py` and `fixedincomelib/product/product_display_visitor.py` before running the examples.

Use the provided `qfCreateProductxxx` APIs to create product objects and `qfDisplayProduct` to inspect their fields.

These objects also form a foundation for a later `value_engine`, which can combine their contract attributes with the necessary rates and market data to calculate cash amounts and present values. The engine reads the product objects directly; the display tables are for inspection.

In [2]:
import sys
import QuantLib as ql

from fixedincomelib import (
    qfCreateProductFixedAccruedCashflow,
    qfCreateProductOvernightIndexCashflow,
    qfDisplayProduct,
)
from fixedincomelib.product import (
    ProductCashflow,
    ProductFixedAccruedCashflow, ProductOvernightIndexCashflow,
)

print(f"Python {sys.version.split()[0]} | QuantLib {ql.__version__}")

Python 3.14.6 | QuantLib 1.43


## 1. Fixed-accrual cashflow

The API returns a `ProductFixedAccruedCashflow` object containing contract fields and an accrual year fraction. `qfDisplayProduct` displays its fields; the fraction is checked below. This class has no fixed-rate field.

In [3]:
fixed = qfCreateProductFixedAccruedCashflow(
    effective_date="2026-01-02",
    termination_date="2026-04-02",
    currency="USD",
    notional=1_000_000,
    accrual_basis="ACT/360",
    payment_date="2026-04-06",
    business_day_convention="F",
    holiday_convention="USGS",
)
assert isinstance(fixed, ProductCashflow)
assert isinstance(fixed, ProductFixedAccruedCashflow)
print("Created:", type(fixed).__name__)
qfDisplayProduct(fixed)

Created: ProductFixedAccruedCashflow


,Name,Value
0,Product Type,PRODUCT_FIXED_ACCRUED
1,Notional,1000000
2,Currency,USD
3,Long Or Short,LONG
4,Effective Date,2026-01-02
5,Termination Date,2026-04-02
6,Accrual Basis,ACT/360
7,Payment Date,2026-04-06
8,Business Day Convention,F
9,Holiday Convention,USGS


In [8]:
# 90 calendar days / 360. This number is a year fraction, not a cash amount.
assert fixed.accrued == 90 / 360
assert fixed.termination_date.ISO() == "2026-04-02"
assert fixed.payment_date.ISO() == "2026-04-06"
assert fixed.first_date == fixed.effective_date
assert fixed.last_date == fixed.payment_date
print(f"Accrual year fraction: {fixed.accrued:.6f}")
print("Contractual accrual end:", fixed.termination_date.ISO())
print("Payment date:", fixed.payment_date.ISO())

Accrual year fraction: 0.250000
Contractual accrual end: 2026-04-02
Payment date: 2026-04-06


## 2. Overnight-index cashflow

The API returns a `ProductOvernightIndexCashflow` object. It resolves the tenor to a termination date, obtains currency from the index, and stores the contract fields.

`0.001` is 10 basis points as an annual decimal rate. Negative notional implies `SHORT`. Creation and display do not calculate an interest amount.

In [5]:
overnight = qfCreateProductOvernightIndexCashflow(
    effective_date="2026-01-02",
    term_or_termination_date="3M",
    overnight_index="SOFR-1B",
    notional=-1_000_000,
    compounding_method="compound",
    spread=0.001,
    payment_date="2026-04-06",
)
assert isinstance(overnight, ProductCashflow)
assert isinstance(overnight, ProductOvernightIndexCashflow)
assert overnight.notional == -1_000_000
assert overnight.currency.value_str == "USD"
print("Created:", type(overnight).__name__)
qfDisplayProduct(overnight)

Created: ProductOvernightIndexCashflow


,Name,Value
0,Product Type,PRODUCT_OVERNIGHT_INDEX_CASHFLOW
1,Notional,-1000000
2,Currency,USD
3,Long Or Short,SHORT
4,Effective Date,2026-01-02
5,Termination Date,2026-04-02
6,ON Index,SOFR-1B
7,Compounding Method,COMPOUND
8,Spread,0.001
9,Payment Date,2026-04-06


In [6]:
# Date and tenor inputs resolve to the same termination date.
overnight_by_date = qfCreateProductOvernightIndexCashflow(
    effective_date="2026-01-02",
    term_or_termination_date="2026-04-02",
    overnight_index="SOFR-1B",
    notional=-1_000_000,
    compounding_method="compound",
    spread=0.001,
    payment_date="2026-04-06",
)
assert overnight.termination_date == overnight_by_date.termination_date
assert overnight.payment_date == overnight_by_date.payment_date
print("Date and tenor inputs agree:", overnight.termination_date.ISO())

Date and tenor inputs agree: 2026-04-02


## 3. Currency and payment defaults

Supported indices are `SOFR-1B`, `FF-1B` and `SONIA-1B` (case-insensitive). Currency comes from the index, and an omitted payment date defaults to termination. Compounding labels are `simple`, `arithmetic` and `compound`.

In [7]:
sonia = qfCreateProductOvernightIndexCashflow(
    effective_date="2026-01-02",
    term_or_termination_date="3M",
    overnight_index="SONIA-1B",
    notional=500_000,
    compounding_method="arithmetic",
)
assert sonia.currency.value_str == "GBP"
assert sonia.payment_date == sonia.termination_date
qfDisplayProduct(sonia)

,Name,Value
0,Product Type,PRODUCT_OVERNIGHT_INDEX_CASHFLOW
1,Notional,500000
2,Currency,GBP
3,Long Or Short,LONG
4,Effective Date,2026-01-02
5,Termination Date,2026-04-02
6,ON Index,SONIA-1B
7,Compounding Method,ARITHMETIC
8,Spread,0.0
9,Payment Date,2026-04-02


In [11]:
from fixedincomelib.date.basics import Date, Period
from fixedincomelib.date.utilities import add_period, accrued
from fixedincomelib.market.basics import AccrualBasis

# --- inputs (from the problem) ---
S, coupon, r_mkt, y_ref = 100.0, 0.04, 0.02, 0.06
today    = Date("2026-01-02")   # a coupon date, so accrued = 0
maturity = Date("2035-01-02")   # exactly 9 years left
cal = ql.NullCalendar()

# 3M delivery date and repo year fraction, using the library helpers
delivery = add_period(today, Period("3M"), ql.Unadjusted, cal)
T = accrued(today, delivery, AccrualBasis("ACT/360"))   # 90/360 = 0.25

# --- the bond ---
schedule = ql.Schedule(today, maturity, ql.Period(ql.Semiannual), cal,
                       ql.Unadjusted, ql.Unadjusted,
                       ql.DateGeneration.Backward, False)
dc = ql.Thirty360(ql.Thirty360.BondBasis)
bond = ql.FixedRateBond(0, 100.0, schedule, [coupon], dc)

# 1) forward clean price
ai_today = bond.accruedAmount(today)       # 0
ai_fwd   = bond.accruedAmount(delivery)    # 1.00 (3 of 6 months of a 2.00 coupon)
BF = (S + ai_today) * (1 + r_mkt * T) - ai_fwd

# 2) conversion factor: clean price at a flat 6% yield, divided by 100
K = bond.cleanPrice(y_ref, dc, ql.Compounded, ql.Semiannual, today) / 100

# 3) normalized forward
print(f"BF = {BF:.4f}, K = {K:.6f}, BF/K = {BF / K:.4f}")

BF = 99.5000, K = 0.862465, BF/K = 115.3670


### Q1(4). What the conversion factor does

Bonds in the delivery basket have different coupons and maturities, so their raw prices are not comparable. The conversion factor K rescales each bond to the price of a hypothetical bond yielding 6%. Dividing a bond's forward price by K puts every bond on one basis: the futures price at which delivering that bond breaks even.

K is built from the clean price, so it ignores accrued interest on purpose. Accrued interest is not normalized; it is paid at its actual value. The invoice amount at delivery is

$$\text{Invoice} = F \times K + \text{Accrued Interest}$$

The seller is therefore compensated for accrued interest exactly, and K only has to equalize the clean-price part. This is also why the forward price BF is quoted clean, so that BF/K is comparable to the quoted futures price F.

K is exact only when yields equal 6%. At other yields it over- or under-values some bonds, which is why one bond ends up cheapest to deliver.

In [10]:
F = 115.3670
r_imp = ((F * K + ai_fwd) / S - 1) / T
print(f"Implied repo = {r_imp:.4%}")   # about 2.00%

Implied repo = 1.9999%


### Q2(2). Choosing the cheapest-to-deliver bond

The CTD is the bond with the **highest implied repo** (equivalently, the lowest BF/K).

The implied repo is the return from buying the bond, financing it, selling the future and delivering. The seller compares it to the general collateral repo rate $r_{mkt}$:

- If $r_{imp} > r_{mkt}$: borrow at $r_{mkt}$, buy the bond, short the future and deliver. This locks in a riskless profit of $r_{imp} - r_{mkt}$. Traders do this until the gap closes, which raises the bond's price or lowers the future's.
- If $r_{imp} < r_{mkt}$: delivering that bond is worse than lending cash at the repo rate, so nobody delivers it.

The seller therefore delivers the basket bond with the highest implied repo, since it costs the least to buy and deliver per unit of invoice value. In a consistent market the CTD's implied repo sits close to $r_{mkt}$. A large gap signals an arbitrage or a pricing error.

### Q3. Scenario analysis

**What can change the CTD over the next N days.**

| Driver | Effect |
|---|---|
| Yield level vs. the 6% notional | Above 6%: CTD tends to be the lowest-duration bond (short maturity, high coupon). Below 6%: CTD tends to be the highest-duration bond (long maturity, low coupon). |
| Curve shape (steepening / flattening) | Long and short bonds reprice differently, which can flip the ranking. |
| Carry and roll-down | Over N days, each bond's coupon income, repo cost and time to maturity change. |

The reason K creates this effect is that it is exact only at a 6% yield. Away from 6%, the bonds that K over-values become cheap to deliver.

**Method.** Choose a scenario (for example a +50bp parallel shift in 30 days), reprice every basket bond at the new yields, recompute $r_{imp}$ (or BF/K) for each, and check whether the bond with the highest implied repo has changed.

**Why the CTD matters.**

- **Hedging:** the futures price tracks the CTD, so the contract's DV01 and hedge ratio are the CTD's. If the CTD switches, a hedge built on the old CTD no longer works.
- **Delivery option:** the short side can choose which bond to deliver. This option has real value, and it is embedded in the futures price.
- **Basis trading:** a basis trade (long bond, short future) is built on the CTD, so identifying it correctly decides which trade is cheapest.